# upper_ppo_direct_last QoS comparison

This notebook compares three controllers on the seven controllable Jain scenarios:

- **Zero action**: no learned/heuristic upper policy.
- **Full heuristic**: rule-based upper expert executed through the same lower directional-offset path.
- **Direct PPO**: `models/upper_ppo_direct_last/run_20260701_232103/upper_ppo_final.zip`.

It plots network QoS, delay, SINR, RSRQ, start/end PRB movement, and per-slice QoS/PRB summaries.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path('train_upper_ppo_3gnb.py').exists():
    os.chdir(Path.cwd().parent)

MODEL_PATH = Path('models/upper_ppo_direct_last/run_20260701_232103/upper_ppo_final.zip')
ZERO_DIR = Path('results/zero_action_baseline')
HEURISTIC_DIR = Path('results/upper_lower_heuristic_baseline')
PPO_DIR = Path('results/upper_ppo_direct_last_eval')
EPISODES_PER_SCENARIO = 10
SEED = 7
RERUN_EVAL = False

if RERUN_EVAL:
    subprocess.run([sys.executable, 'run_zero_action_baseline.py', '--out-dir', str(ZERO_DIR), '--episodes-per-scenario', str(EPISODES_PER_SCENARIO), '--seed', str(SEED)], check=True)
    subprocess.run([sys.executable, 'run_heuristic_baseline.py', '--out-dir', str(HEURISTIC_DIR), '--episodes-per-scenario', str(EPISODES_PER_SCENARIO), '--seed', str(SEED)], check=True)
    subprocess.run([sys.executable, 'run_trained_policy_eval.py', '--model-path', str(MODEL_PATH), '--out-dir', str(PPO_DIR), '--episodes-per-scenario', str(EPISODES_PER_SCENARIO), '--seed', str(SEED)], check=True)

for path in [MODEL_PATH, ZERO_DIR, HEURISTIC_DIR, PPO_DIR]:
    assert path.exists(), path

print('Model:', MODEL_PATH)
print('Zero traces:', ZERO_DIR)
print('Heuristic traces:', HEURISTIC_DIR)
print('PPO traces:', PPO_DIR)


In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SCENARIOS = [
    'jain_balance_controllable',
    'jain_control_urllc',
    'jain_control_mmtc',
    'jain_control_embb_urllc',
    'jain_control_embb_mmtc',
    'jain_control_urllc_mmtc',
    'jain_control_outer_congested',
    'jain_control_mixed',
]
SLICES = ['eMBB', 'URLLC', 'mMTC']
CONTROLLERS = ['Zero action', 'Full heuristic', 'Direct PPO']
COLORS = {'Zero action': '#4c78a8', 'Full heuristic': '#54a24b', 'Direct PPO': '#f58518'}

def load_controller(pattern, label):
    rows = []
    frames = {}
    for path in sorted(glob.glob(pattern)):
        df = pd.read_csv(path)
        scenario = str(df['scenario_name'].iloc[0])
        frames[scenario] = df
        row = {
            'controller': label,
            'scenario': scenario,
            'throughput_mbps': df['network_throughput_mbps'].mean(),
            'offered_mbps': df['network_offered_mbps'].mean(),
            'delivery_ratio': df['network_delivery_ratio'].mean(),
            'completed_delay_ms': df['network_completed_delay_ms'].mean(),
            'mean_hol_delay_ms': df['network_mean_hol_delay_ms'].mean(),
            'max_hol_delay_ms': df['network_max_hol_delay_ms'].mean(),
            'queue_kbits': df['network_queue_kbits'].mean(),
            'drop_ratio': df['network_drop_ratio'].mean(),
            'packet_failure_ratio': df['network_packet_failure_ratio'].mean(),
            'sla_severity': df['sla_severity'].mean(),
            'sla_count': df['sla_count'].mean(),
            'handover_count': df['handover_count'].mean(),
            'jain_fairness_raw': df['jain_fairness_raw'].mean(),
            'demand_load_std_start': df['gnb_total_demand_load_std_start'].mean(),
            'demand_load_std_end': df['gnb_total_demand_load_std_end'].mean(),
        }
        for st in SLICES:
            row[f'{st}_throughput_mbps'] = df[f'qos_slice_throughput_mbps_{st}'].mean()
            row[f'{st}_delivery_ratio'] = df[f'qos_slice_delivery_ratio_{st}'].mean()
            row[f'{st}_mean_hol_delay_ms'] = df[f'qos_slice_mean_hol_delay_ms_{st}'].mean()
            row[f'{st}_queue_kbits'] = df[f'qos_slice_queue_kbits_{st}'].mean()
            row[f'{st}_sinr_db'] = df[f'qos_slice_sinr_db_{st}'].mean()
            row[f'{st}_rsrq_db'] = df[f'qos_slice_rsrq_db_{st}'].mean()
            row[f'{st}_demand_start_prb'] = df[f'slice_demand_prb_start_{st}'].mean()
            row[f'{st}_demand_end_prb'] = df[f'slice_demand_prb_end_{st}'].mean()
            row[f'{st}_used_start_prb'] = df[f'slice_used_prb_start_{st}'].mean()
            row[f'{st}_used_end_prb'] = df[f'slice_used_prb_end_{st}'].mean()
        rows.append(row)
    return pd.DataFrame(rows), frames

zero_summary, zero_frames = load_controller(str(ZERO_DIR / '*_zero_action_trace.csv'), 'Zero action')
heuristic_summary, heuristic_frames = load_controller(str(HEURISTIC_DIR / '*_heuristic_trace.csv'), 'Full heuristic')
ppo_summary, ppo_frames = load_controller(str(PPO_DIR / '*_trained_policy_trace.csv'), 'Direct PPO')

summary = pd.concat([zero_summary, heuristic_summary, ppo_summary], ignore_index=True)
summary['scenario'] = pd.Categorical(summary['scenario'], categories=SCENARIOS, ordered=True)
summary['controller'] = pd.Categorical(summary['controller'], categories=CONTROLLERS, ordered=True)
summary = summary.sort_values(['scenario', 'controller']).reset_index(drop=True)
summary.to_csv(PPO_DIR / 'qos_three_controller_summary.csv', index=False)

frames_by_controller = {'Zero action': zero_frames, 'Full heuristic': heuristic_frames, 'Direct PPO': ppo_frames}
print(summary.shape)
summary.round(3)


In [ ]:
wide = summary.pivot(index='scenario', columns='controller')
wide[[
    'throughput_mbps',
    'delivery_ratio',
    'mean_hol_delay_ms',
    'queue_kbits',
    'sla_severity',
    'handover_count',
    'jain_fairness_raw',
]].round(3)


In [ ]:
x = np.arange(len(SCENARIOS))
width = 0.25
fig, axes = plt.subplots(2, 4, figsize=(22, 10), constrained_layout=True)
metrics = [
    ('throughput_mbps', 'Throughput (Mbps)', axes[0, 0], None),
    ('delivery_ratio', 'Delivery ratio', axes[0, 1], (0, 1.05)),
    ('mean_hol_delay_ms', 'Mean HOL delay (ms)', axes[0, 2], None),
    ('max_hol_delay_ms', 'Max HOL delay (ms)', axes[0, 3], None),
    ('queue_kbits', 'Queue (kbits, log scale)', axes[1, 0], 'symlog'),
    ('sla_severity', 'SLA severity', axes[1, 1], None),
    ('handover_count', 'Handovers/window', axes[1, 2], None),
    ('jain_fairness_raw', 'Jain fairness', axes[1, 3], (0, 1.05)),
]
for metric, title, ax, ylim in metrics:
    for idx, controller in enumerate(CONTROLLERS):
        vals = [summary[(summary['scenario'] == s) & (summary['controller'] == controller)][metric].mean() for s in SCENARIOS]
        ax.bar(x + (idx - 1) * width, vals, width, label=controller, color=COLORS[controller])
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(SCENARIOS, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.25)
    if ylim == 'symlog':
        ax.set_yscale('symlog', linthresh=1.0)
    elif ylim is not None:
        ax.set_ylim(*ylim)
axes[0, 0].legend(loc='best')
fig.suptitle('Network QoS: zero vs full heuristic vs direct PPO', fontsize=14)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)
for ax, metric, title in [
    (axes[0], 'sinr_db', 'Per-slice average SINR (dB)'),
    (axes[1], 'rsrq_db', 'Per-slice average RSRQ (dB)'),
]:
    labels = []
    positions = []
    pos = 0
    for scenario in SCENARIOS:
        for st in SLICES:
            labels.append(f'{scenario}\n{st}')
            positions.append(pos)
            for idx, controller in enumerate(CONTROLLERS):
                row = summary[(summary['scenario'] == scenario) & (summary['controller'] == controller)].iloc[0]
                ax.bar(pos + (idx - 1) * 0.22, row[f'{st}_{metric}'], 0.22, color=COLORS[controller], label=controller if pos == 0 else None)
            pos += 1
        pos += 0.6
    ax.set_title(title)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=70, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(len(SLICES), 3, figsize=(18, 12), constrained_layout=True)
per_slice_metrics = [
    ('throughput_mbps', 'Throughput (Mbps)', None),
    ('delivery_ratio', 'Delivery ratio', (0, 1.05)),
    ('mean_hol_delay_ms', 'Mean HOL delay (ms)', None),
]
for r, st in enumerate(SLICES):
    for c, (metric, title, ylim) in enumerate(per_slice_metrics):
        ax = axes[r, c]
        for idx, controller in enumerate(CONTROLLERS):
            vals = [summary[(summary['scenario'] == s) & (summary['controller'] == controller)].iloc[0][f'{st}_{metric}'] for s in SCENARIOS]
            ax.plot(SCENARIOS, vals, marker='o', lw=2, label=controller, color=COLORS[controller])
        ax.set_title(f'{st}: {title}')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(alpha=0.25)
        if ylim is not None:
            ax.set_ylim(*ylim)
        if r == 0 and c == 0:
            ax.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(len(SCENARIOS), 2, figsize=(15, 3.0 * len(SCENARIOS)), constrained_layout=True)
controller = 'Direct PPO'
for row_idx, scenario in enumerate(SCENARIOS):
    df = frames_by_controller[controller][scenario]
    demand_start = np.asarray([df[f'gnb_demand_prb_start_g{i}'].mean() for i in range(3)])
    demand_end = np.asarray([df[f'gnb_demand_prb_end_g{i}'].mean() for i in range(3)])
    used_start = np.asarray([df[f'gnb_used_prb_start_g{i}'].mean() for i in range(3)])
    used_end = np.asarray([df[f'gnb_used_prb_end_g{i}'].mean() for i in range(3)])
    g = np.arange(3)
    ax = axes[row_idx, 0]
    ax.bar(g - 0.18, demand_start, 0.36, label='start', color='#72b7b2')
    ax.bar(g + 0.18, demand_end, 0.36, label='end', color='#e45756')
    ax.set_title(f'{scenario}: Direct PPO demand PRB start/end')
    ax.set_xticks(g)
    ax.set_xticklabels(['g0', 'g1', 'g2'])
    ax.grid(axis='y', alpha=0.25)
    if row_idx == 0:
        ax.legend()
    ax = axes[row_idx, 1]
    ax.bar(g - 0.18, used_start, 0.36, label='start', color='#72b7b2')
    ax.bar(g + 0.18, used_end, 0.36, label='end', color='#e45756')
    ax.set_title(f'{scenario}: Direct PPO used PRB start/end')
    ax.set_xticks(g)
    ax.set_xticklabels(['g0', 'g1', 'g2'])
    ax.grid(axis='y', alpha=0.25)
    if row_idx == 0:
        ax.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(len(SLICES), 2, figsize=(16, 10), constrained_layout=True)
for r, st in enumerate(SLICES):
    for c, phase in enumerate(['demand', 'used']):
        ax = axes[r, c]
        for controller in CONTROLLERS:
            starts = [summary[(summary['scenario'] == s) & (summary['controller'] == controller)].iloc[0][f'{st}_{phase}_start_prb'] for s in SCENARIOS]
            ends = [summary[(summary['scenario'] == s) & (summary['controller'] == controller)].iloc[0][f'{st}_{phase}_end_prb'] for s in SCENARIOS]
            ax.plot(SCENARIOS, ends, marker='o', lw=2, label=f'{controller} end', color=COLORS[controller])
            if controller == 'Direct PPO':
                ax.plot(SCENARIOS, starts, marker='x', lw=1.2, ls='--', color='black', label='Direct PPO start')
        ax.set_title(f'{st}: slice {phase} PRB')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(alpha=0.25)
        if r == 0 and c == 0:
            ax.legend(fontsize=8)
plt.show()


## Rendered plots

The PNGs below were generated from the current refreshed traces.

![Network QoS comparison](../results/upper_ppo_direct_last_eval/qos_three_controller.png)

![SINR and RSRQ by slice](../results/upper_ppo_direct_last_eval/radio_sinr_rsrq_by_slice.png)

![Per-slice QoS](../results/upper_ppo_direct_last_eval/per_slice_qos.png)

![Direct PPO PRB start/end](../results/upper_ppo_direct_last_eval/direct_ppo_prb_start_end_by_scenario.png)

![Per-slice PRB](../results/upper_ppo_direct_last_eval/per_slice_prb.png)


## Quick read

The full heuristic is now included as the middle reference. `upper_ppo_direct_last` is very aggressive: it often gives excellent Jain/load balance, but several multi-slice cases pay with worse delivery, queue, HOL delay, and SLA severity. The per-slice and radio plots make clear which slice absorbs that cost.
